# Use the Evaluator-Optimizer Agent design pattern that can draft LinkedIn connection messages

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)

In [ ]:
google_api_key = os.getenv('GEMINI_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
request = "Please draft a LinkedIn connection message highlighting my 3 years of experience in the Data Science and Analytics field and express interest to connect and learn about open roles in their company"
request += "Do not add any place holders at any point and consider the character limit to be 300 characters."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
models = []
responses = []

In [ ]:
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
model_name = "gemini-2.5-flash"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
models.append(model_name)
responses.append(answer)

In [ ]:
model_name = "gemini-2.5-flash-lite"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
models.append(model_name)
responses.append(answer)

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
models.append(model_name)
responses.append(answer)

In [ ]:
print(models)
print(responses)

In [ ]:
# It's nice to know how to use "zip"
for model, response in zip(models, responses):
    print(f"Competitor: {model}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(responses):
    together += f"# Response from model {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a linkedin connection messages between {len(models)} models.
Each model has been given this request:

{request}

Your job is to evaluate each response for clarity and ability of the models to follow instructions and rank them in order of best to worst.
If there are placehodlers or if the character limit has exceeded the instructions rank those models very low.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the models, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = models[int(result)-1]
    print(f"Rank {index+1}: {competitor}")